# Baseline Mock 실행

모델 없이 입력 로딩, JSON 출력, `submission.csv` 생성과 형식 검증만 확인한다.

### 대회 데이터 구조 확인
- 데이터 파일 불러오기 
- 데이터 구조 유형 파악
- 문서 및 라벨 개수 파악
- 모델 구축 시 데이터 사용 방법 구축

In [ ]:
from pathlib import Path
import shutil, zipfile

ZIP_PATH = Path('../data/open.zip') # 원본파일 위치 확인
WORK = Path('/.work/mock_baseline') # 
if WORK.exists():
    shutil.rmtree(WORK)
(WORK / 'data').mkdir(parents=True) #원본파일 수정 방지를 위해서 work 폴더 생성 
(WORK / 'baseline').mkdir() # 

with zipfile.ZipFile(ZIP_PATH) as z:
    members = ['data/test.jsonl.gz', 'data/항목표.json', 'data/정답스키마_디코딩.json', 'baseline/script.py']
    for member in members:
        target = WORK / member
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(z.read(member)) # 테스트 데이터 복사 후 작업준비
print('준비 완료:', WORK)

준비 완료: \.work\mock_baseline


In [ ]:
import os, subprocess, sys

result = subprocess.run( 
    [sys.executable, "baseline/script.py", "--mock"], #.py  실행
    cwd=WORK,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print("returncode:", result.returncode) #유니코드 확인
print(result.stdout)
print(result.stderr or "")
assert result.returncode == 0

returncode: 0

[baseline] 입력 10건 ← ./data\test.jsonl.gz
[baseline] 모델 로드 0.0s
[baseline] 프롬프트 토큰 중앙값 2,962 · 최대 2,999 · 예산 축소 0건
[baseline]   10/10건 … 0s
[baseline] {"건수": 10, "모델로드_s": 0.0, "추론_s": 0.0, "건당_s": 0.0, "전체_s": 0.1, "유효JSON": 10, "메운_항목수": 0, "근거_유지": 0, "근거_원문불일치_폐기": 0, "출력": "./output\\submission.csv", "자가검증": "PASS"}



In [8]:
import csv

out = WORK / 'output' / 'submission.csv'
print('출력:', out, out.exists())
with out.open(encoding='utf-8', newline='') as f:
    rows = list(csv.reader(f))
print('shape:', len(rows)-1, 'x', len(rows[0]))
print('header:', rows[0])
assert len(rows[0]) == 49
assert len(rows) == 11
print('MOCK_SUBMISSION_FORMAT_OK')

출력: \.work\mock_baseline\output\submission.csv True
shape: 10 x 49
header: ['id', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9', 'v10', 'v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19', 'v20', 'v21', 'v22', 'v23', 'v24', 'e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'e7', 'e8', 'e9', 'e10', 'e11', 'e12', 'e13', 'e14', 'e15', 'e16', 'e17', 'e18', 'e19', 'e20', 'e21', 'e22', 'e23', 'e24']
MOCK_SUBMISSION_FORMAT_OK


## 주의

`--mock`은 모델의 정확도를 검증하지 않는다. 실제 추론은 평가 서버의 고정 모델·vLLM 환경이 필요하다. 이 Notebook은 로컬 형식과 실행 흐름만 확인한다.